# 01 - A LangChain agent, traced with OpenTelemetry

Goal: run a LangChain agent that sends its traces to LangSmith through the **OpenTelemetry SDK**, then measure what those traces carry against the 83 Phase 2 attributes.

The agent is a bank support assistant with a knowledge-base retriever and an account-lookup tool, running on the model configured in `.env` (Claude on Amazon Bedrock by default, or an open-source model). It is provided. This track is about the metadata on its traces:
the next challenges add it **before the spans leave the process**, not after they land in LangSmith.
Everything lands in **your** LangSmith workspace. Run the cells top to bottom.

## Step 1: Enter your LangSmith key

The cell asks for a [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key) and a short label for your project.
The key is written to a local `.env` file (git-ignored) and never printed.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import sys
sys.path.insert(0, str(ROOT / "helpers"))
from lab_credentials import collect
collect()

In [ ]:
import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

## Step 2: How the agent's spans reach LangSmith

LangChain emits OpenTelemetry spans when `LANGSMITH_OTEL_ENABLED=true`. They go to whatever global `TracerProvider` exists when the agent is first built, so the provider is configured **first**.
`otel_setup.configure()` does that. Read it: later challenges hang their enrichment on exactly this provider.

In [ ]:
print(inspect.getsource(otel_setup.configure))
print(inspect.getsource(otel_setup.to_langsmith_attributes))

## Step 3: Run the agent

`configure()` with no arguments is the default: no resource attributes, no processors. Then build the agent and ask it one question. `bank_support_turn` is the root span the agent code opens around each turn.

In [ ]:
otel_setup.configure()
agent = bank_agent.build_agent()
print(bank_agent.ask(agent, "What is the daily PayNow transfer limit on my savings account?"))

Send three more turns, including one that needs the account tool, then wait for ingestion. OTLP is asynchronous, so this takes up to two minutes.

In [ ]:
bank_traffic.run_turns(bank_agent, agent, [
    "How do I reset my card PIN?",
    "What is the daily limit on account SAV-4821?",
    "Transfer SGD 500 from my savings to card 4111 1111 1111 1111.",
])
otel_setup.flush()
n = bank_traffic.wait_for_runs(PROJECT, minimum=4)
print(f"{n} root runs have arrived in {PROJECT}")
lab_links.project_link(hint="open a trace: the tree, run types and Metadata tab show what OTel carries by default")

## Step 4: Read one trace

Print one trace as a tree: span name, run type, tags and the metadata keys that arrived. Note what is **not** there: environment, channel, geography, risk.

In [ ]:
runs = enrich.project_runs(client, PROJECT)
enrich.show_trace(enrich.agent_runs(runs))

## Step 5: Build the gap report

Each of the 83 attributes is marked **present**, **missing**, or **reference** (not built hands-on). The `stage` column says where it is expected:

- `native` - LangChain's OTel spans should carry it.
- `2`..`7` - you add it in that challenge: 2 resource attributes, 3 SpanProcessor and baggage, 4 span attributes in tool code, 5 geography, 6 evaluator, 7 annotation queue.
- `ref` - reference only.

In [ ]:
report = coverage.gap_report(client, PROJECT, runs=runs, write=True)
report[["#", "attribute", "stage", "status", "where"]]

In [ ]:
coverage.summary(report)

In [ ]:
native_missing = report[(report["stage"] == "native") & (report["status"] == "missing")]
print(f"{len(native_missing)} attributes the OTel spans should carry are missing:")
native_missing[["#", "attribute", "where"]]

A `native` row that is `missing` is a finding about what the integration emits by default; the numbered stages are what challenges 2 to 7 fill in.

## Step 6: See it in LangSmith, and verify

In [ ]:
lab_links.project_link(hint="open one trace and compare its Metadata tab with the gap report")

In [ ]:
!verify-connection

Next: challenge 2 adds deployment and identity as OTel resource attributes.